In [1]:
# 다양한 파일 처리

In [2]:
import pandas as pd
import json
from pathlib import Path

In [3]:
base_dir = Path(".") # . 은 현재 폴더
raw_dir = base_dir / "data" / "raw" # 원본 데이터 저장 위치
interim_dir = base_dir / "data" / "interim"

In [4]:
# make directory ==> 모음을 없애고 줄인다. 리눅스 명령어와 비슷
raw_dir.mkdir(parents=True, exist_ok=True) # data/raw 생성
# parents=True 는 부모 폴더를 먼저 만든다.
# exist_ok=True 는 존재하면 에러 내지말고 사용
interim_dir.mkdir(parents=True, exist_ok=True) # data/interim 생성

In [5]:
main_file = interim_dir / "ai4i_cleaned.csv" # data/interim/ai4i_cleaned.csv
df_main = pd.read_csv(main_file)

In [6]:
df_main.shape

(10000, 18)

In [7]:
display(df_main.head())

,udi,product_id,type,air_temp_k,process_temp_k,rotational_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,hdf,pwf,osf,rnf,temp_diff_k,power_index,tool_wear_level,tool_wear_outlier_flag
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,10.5,66382.8,low,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,10.5,65190.4,low,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,10.4,74001.2,low,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,10.4,56603.5,low,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,10.5,56320.0,low,0


In [8]:
equipment_file = raw_dir / 'equipment_master.csv'
df_equipment = pd.read_csv(equipment_file)
df_equipment

,type,type_desc,maintenance_cycle_days,inspection_level
0,L,low grade,30,basic
1,M,medium grade,20,standard
2,H,high grade,10,strict


In [9]:
# 미션 1 : 메인(df_main)에 Left Join 을 장비(df_equipment) 붙이기

In [10]:
df_merged = pd.merge(df_main, df_equipment, on="type", how="left") # SQL Left Join
df_merged

,udi,product_id,type,air_temp_k,process_temp_k,rotational_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,...,pwf,osf,rnf,temp_diff_k,power_index,tool_wear_level,tool_wear_outlier_flag,type_desc,maintenance_cycle_days,inspection_level
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0,0,0,10.5,66382.8,low,0,medium grade,20,standard
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,0,0,0,10.5,65190.4,low,0,low grade,30,basic
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,0,0,0,10.4,74001.2,low,0,low grade,30,basic
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,0,0,0,10.4,56603.5,low,0,low grade,30,basic
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,0,0,0,10.5,56320.0,low,0,low grade,30,basic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,M24855,M,298.8,308.4,1604,29.5,14,0,0,...,0,0,0,9.6,47318.0,low,0,medium grade,20,standard
9996,9997,H39410,H,298.9,308.4,1632,31.8,17,0,0,...,0,0,0,9.5,51897.6,low,0,high grade,10,strict
9997,9998,M24857,M,299.0,308.6,1645,33.4,22,0,0,...,0,0,0,9.6,54943.0,low,0,medium grade,20,standard
9998,9999,H39412,H,299.0,308.7,1408,48.5,25,0,0,...,0,0,0,9.7,68288.0,low,0,high grade,10,strict


In [11]:
# 미션 2 : 폴더 안 여러 CSV 읽어서 하나로 합치기. 비유 : 01월.xlsx~12월.xlsx 합치기

In [12]:
daily_dir = raw_dir / "daily_targets"
file_list = list( daily_dir.glob("*.csv") ) # 모든 CSV 파일
file_list # [ ~a.csv, ~b.csv, ~c.csv ]

[WindowsPath('data/raw/daily_targets/target_a.csv'),
 WindowsPath('data/raw/daily_targets/target_b.csv'),
 WindowsPath('data/raw/daily_targets/target_c.csv')]

In [13]:
df_list = [] # 비어있는 리스트 준비
for file in file_list: # [ ~a.csv, ~b.csv, ~c.csv ]
    df_temp = pd.read_csv(file) # ~.csv 파일 읽기
    df_list.append(df_temp) # 리스트에 추가

In [14]:
df_targets = pd.concat(df_list, ignore_index=True) # 원래 인덱스 무시 --> 새 인덱스
df_targets

,type,shift,daily_target
0,L,A,120
1,M,A,150
2,H,A,180
3,L,B,110
4,M,B,145
5,H,B,175
6,L,C,100
7,M,C,140
8,H,C,170


In [15]:
# 미션 3 : ~.json(json 파일)을 읽어서 DataFrame으로 바꿔서 합친다.

In [16]:
json_file = raw_dir / "api_sample.json"
# with 절을 사용하면 자동 close 됨
with open(json_file, "r", encoding="utf-8") as f:
    api_data = json.load(f)
    print(api_data)

[{'type': 'L', 'recommended_temp_range': '295~305K', 'maintenance_priority': 'low'}, {'type': 'M', 'recommended_temp_range': '300~310K', 'maintenance_priority': 'medium'}, {'type': 'H', 'recommended_temp_range': '305~315K', 'maintenance_priority': 'high'}]


In [17]:
df_api = pd.DataFrame(api_data)
df_api

,type,recommended_temp_range,maintenance_priority
0,L,295~305K,low
1,M,300~310K,medium
2,H,305~315K,high


In [18]:
df_merged = pd.merge(df_merged, df_api, on="type", how="left")
df_merged.head(3)

,udi,product_id,type,air_temp_k,process_temp_k,rotational_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,...,rnf,temp_diff_k,power_index,tool_wear_level,tool_wear_outlier_flag,type_desc,maintenance_cycle_days,inspection_level,recommended_temp_range,maintenance_priority
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0,10.5,66382.8,low,0,medium grade,20,standard,300~310K,medium
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,0,10.5,65190.4,low,0,low grade,30,basic,295~305K,low
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,0,10.4,74001.2,low,0,low grade,30,basic,295~305K,low


In [19]:
# end